<a href="https://colab.research.google.com/github/topiltzin/test-repo/blob/master/Inegi_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip -q install -U transformers accelerate "pandas==2.2.3" "requests==2.32.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 7.0 MB/s eta 0:00:00


In [4]:
from google.colab import userdata
from pathlib import Path
import requests
import pandas as pd
import json
import os
from datetime import datetime, timezone

INEGI_TOKEN = userdata.get("INEGI_TOKEN")

if not INEGI_TOKEN:
    raise ValueError(
        "Add INEGI_TOKEN to Colab Secrets and enable notebook access."
    )

PROJECT_DIR = Path("/content/inegi_qwen_business")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Project initialized.")

Project initialized.


In [5]:
import requests
from urllib.parse import quote

DENUE_BASE = (
    "https://www.inegi.org.mx/app/api/denue/v1/consulta"
)

def denue_get(path, timeout=60):
    """Call an official DENUE endpoint without exposing the token."""
    url = f"{DENUE_BASE}/{path}/{INEGI_TOKEN}"

    response = requests.get(url, timeout=timeout)
    response.raise_for_status()

    try:
        return response.json()
    except ValueError as exc:
        raise RuntimeError(
            "DENUE returned a response that was not valid JSON. "
            "Check the endpoint and token."
        ) from exc

In [6]:
def denue_count(activity="0", geography="0", size="0"):
    """
    Count establishments by economic activity, geography and size.

    activity:
        "0" = all activities, or a documented activity code.
    geography:
        "0" = national level, or a documented geographic code.
    size:
        "0" = all employment-size categories.
    """
    path = (
        f"Cuantificar/{quote(str(activity))}/"
        f"{quote(str(geography))}/{quote(str(size))}"
    )

    return denue_get(path)


count_data = denue_count()

print("Response type:", type(count_data).__name__)
print(json.dumps(count_data, ensure_ascii=False, indent=2)[:3000])

Response type: list
[
  {
    "AE": "11",
    "AG": "01",
    "Total": "49"
  },
  {
    "AE": "111",
    "AG": "01",
    "Total": "0"
  },
  {
    "AE": "1111",
    "AG": "01",
    "Total": "0"
  },
  {
    "AE": "11111",
    "AG": "01",
    "Total": "0"
  },
  {
    "AE": "11112",
    "AG": "01",
    "Total": "0"
  },
  {
    "AE": "11113",
    "AG": "01",
    "Total": "0"
  },
  {
    "AE": "11114",
    "AG": "01",
    "Total": "0"
  },
  {
    "AE": "11115",
    "AG": "01",
    "Total": "0"
  },
  {
    "AE": "11116",
    "AG": "01",
    "Total": "0"
  },
  {
    "AE": "11119",
    "AG": "01",
    "Total": "0"
  },
  {
    "AE": "1112",
    "AG": "01",
    "Total": "0"
  },
  {
    "AE": "11121",
    "AG": "01",
    "Total": "0"
  },
  {
    "AE": "1113",
    "AG": "01",
    "Total": "0"
  },
  {
    "AE": "11131",
    "AG": "01",
    "Total": "0"
  },
  {
    "AE": "11132",
    "AG": "01",
    "Total": "0"
  },
  {
    "AE": "11133",
    "AG": "01",
    "Total": "0"
  },
  {
    "

In [7]:
if isinstance(count_data, list) and count_data:
    counts_df = pd.DataFrame(count_data)
    display(counts_df.head(20))
else:
    print("No count rows returned. Inspect the response and API documentation.")

,AE,AG,Total
0,11,01,49
1,111,01,0
2,1111,01,0
3,11111,01,0
4,11112,01,0
5,11113,01,0
6,11114,01,0
7,11115,01,0
8,11116,01,0
9,11119,01,0


In [8]:
def denue_search(term, state_code, start=1, end=10):
    """
    Search DENUE by condition and state.
    state_code must be a two-digit official state code.
    """
    if not term or not term.strip():
        raise ValueError("Enter a search term.")

    if not (len(state_code) == 2 and state_code.isdigit()):
        raise ValueError("Use a two-digit state code.")

    if start < 1 or end < start or end - start > 99:
        raise ValueError("Request a range of at most 100 records.")

    path = (
        f"BuscarEntidad/{quote(term.strip(), safe='')}/"
        f"{state_code}/{start}/{end}"
    )

    result = denue_get(path)

    if isinstance(result, list):
        return pd.DataFrame(result)

    return pd.DataFrame()


# Example query: search for restaurants in Jalisco (state code 14).
restaurants = denue_search("restaurantes", "14", 1, 10)

display(restaurants.head(10))

,CLEE,Id,Nombre,Razon_social,Clase_actividad,Estrato,Tipo_vialidad,Calle,Num_Exterior,Num_Interior,...,Ubicacion,Telefono,Correo_e,Sitio_internet,Tipo,Longitud,Latitud,tipo_corredor_industrial,nom_corredor_industrial,numero_local
0,14120722517003411000000000U6,1821892,1 POLLO ASADO,,Restaurantes con servicio de preparación de pi...,0 a 5 personas,CALLE,PUERTO GUAYMAS,68,,...,"ZAPOPAN,Zapopan,JALISCO",,MIGUEL_PENILLAS@HOTMAIL.COM,,Fijo,-103.44957814,20.64273311,,,
1,14039722517000513010000000U4,6206244,1/4 DE KILO HAMBURGUEZAS,,Restaurantes de autoservicio,11 a 30 personas,CALLE,FLORENCIA,2818,9,...,"GUADALAJARA,Guadalajara,JALISCO",,,,Fijo,-103.38380441,20.69488861,CENTRAL DE ABASTO,SIN NOMBRE,11
2,14039722518009561000000000U2,11482779,10 COCINA,,Restaurantes que preparan otro tipo de aliment...,0 a 5 personas,CALLE,MONTEOLIVETE,413,,...,"GUADALAJARA,Guadalajara,JALISCO",3329619266,,,Fijo,-103.33275657,20.70245516,,,
3,14067722514009811000000000U1,9929901,100 VEGANA TAQUERIA,,Restaurantes con servicio de preparación de ta...,0 a 5 personas,CALLE,NICARAGUA,391,,...,"PUERTO VALLARTA,Puerto Vallarta,JALISCO",,,,Fijo,-105.22827198,20.61949230,,,
4,14039722219010481002000000U5,1837664,1000 CAGUAMAS,MIL CAGUAS,Restaurantes con servicio de preparación de al...,6 a 10 personas,AVENIDA,UNION,163,2,...,"GUADALAJARA,Guadalajara,JALISCO",3336156767,,1000CAGUAMAS.COM.MX,Fijo,-103.37350707,20.67323293,EDIFICIO COMERCIAL Y DE SERVICIOS,TORRE UNION,T
5,14098722511002183000000000U3,9508288,1000 CAGUAMAS,,Restaurantes con servicio de preparación de al...,11 a 30 personas,CALLE,INDEPENDENCIA EJE,373,,...,"TLAQUEPAQUE,San Pedro Tlaquepaque,JALISCO",3310292503,TLAQUEPAQUE@1000CAGUAMAS.COM,WWW.1000CAGUAMAS.COM.MX,Fijo,-103.31724920,20.63931617,,,
6,14120722211001223022003286S7,6343317,11008 DOMINOS PIZZA GUADALAJARA CHAPALITA,OPERADORA Y PROCESADORA DE PRODUCTOS DE PANIFI...,Restaurantes de autoservicio,11 a 30 personas,AVENIDA,GUADALUPE,1626,,...,"ZAPOPAN,Zapopan,JALISCO",,DOMINOSPIZZA@ALSEA.NET,WWW.DOMINOS.COM.MX,Fijo,-103.40762829,20.66551645,,,
7,14067722211000373023003286S4,6344394,11014 DOMINOS PIZZA VALLARTA CARACOL,OPERADORA Y PROCESADORA DE PRODUCTOS DE PANIFI...,Restaurantes de autoservicio,11 a 30 personas,AVENIDA,LOS TULES,2180,,...,"PUERTO VALLARTA,Puerto Vallarta,JALISCO",,DOMINOSPIZZA@ALSEA.NET,WWW.DOMINOS.COM.MX,Fijo,-105.23161524,20.64190340,,,
8,14039722211002253022003286S5,6344383,11020 DOMINOS PIZZA GUADALAJARA PROVIDENCIA,OPERADORA Y PROCESADORA DE PRODUCTOS DE PANIFI...,Restaurantes de autoservicio,11 a 30 personas,AVENIDA,PABLO NERUDA,3006,,...,"GUADALAJARA,Guadalajara,JALISCO",,DOMINOSPIZZA@ALSEA.NET,WWW.DOMINOS.COM.MX,Fijo,-103.38723097,20.69790035,,,
9,14039722211002153002003286S7,6344378,11035 DOMINOS PIZZA GUADALAJARA RESIDENCIAL VI...,OPERADORA Y PROCESADORA DE PRODUCTOS DE PANIFI...,Restaurantes de autoservicio,11 a 30 personas,AVENIDA,LAPISLÁZULI,2653,,...,"GUADALAJARA,Guadalajara,JALISCO",,DOMINOSPIZZA@ALSEA.NET,WWW.DOMINOS.COM.MX,Fijo,-103.39330193,20.64174262,,,


In [10]:
!pip -q install scikit-learn

In [11]:
documents = [
    {
        "id": "denue_overview",
        "title": "Qué es el DENUE",
        "text": (
            "El Directorio Estadístico Nacional de Unidades Económicas, "
            "DENUE, ofrece información de establecimientos económicos "
            "en México. Incluye identificación, ubicación, actividad "
            "económica y estrato de personal ocupado. No debe interpretarse "
            "automáticamente como un registro de ingresos, ventas o utilidades."
        ),
        "source": "https://www.inegi.org.mx/servicios/api_denue.html"
    },
    {
        "id": "denue_count",
        "title": "Conteo de establecimientos",
        "text": (
            "El método Cuantificar de la API DENUE permite contar "
            "establecimientos por actividad económica, área geográfica "
            "y estrato de personal ocupado. Se deben registrar los "
            "parámetros utilizados para interpretar correctamente el resultado."
        ),
        "source": "https://www.inegi.org.mx/servicios/api_denue.html"
    },
    {
        "id": "denue_search",
        "title": "Búsqueda de establecimientos",
        "text": (
            "El método BuscarEntidad permite buscar establecimientos "
            "por una condición y limitar la búsqueda a una entidad federativa. "
            "Los resultados pueden incluir nombre, actividad económica, "
            "ubicación y estrato de personal ocupado."
        ),
        "source": "https://www.inegi.org.mx/servicios/api_denue.html"
    },
    {
        "id": "employment_size",
        "title": "Estratos de personal ocupado",
        "text": (
            "La documentación de la API DENUE describe estratos de personal "
            "ocupado. Entre ellos están de 0 a 5, de 6 a 10, de 11 a 30, "
            "de 31 a 50, de 51 a 100, de 101 a 250 y de 251 personas en adelante. "
            "El estrato es una categoría de tamaño, no una medición de ventas "
            "ni de rentabilidad."
        ),
        "source": "https://www.inegi.org.mx/servicios/api_denue.html"
    },
    {
        "id": "limitations",
        "title": "Limitaciones e interpretación",
        "text": (
            "Antes de comparar estadísticas de establecimientos, verifica "
            "la fecha de referencia, cobertura geográfica, definición, "
            "clasificación económica y unidad de medida. Una cantidad de "
            "establecimientos no equivale necesariamente a cantidad de "
            "empresas, trabajadores, ventas o contribución al PIB."
        ),
        "source": "https://www.inegi.org.mx/servicios/api_denue.html"
    }
]

print(f"Documentos en la base inicial: {len(documents)}")

Documentos en la base inicial: 5


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(
    strip_accents="unicode",
    lowercase=True,
    ngram_range=(1, 2)
)

document_matrix = vectorizer.fit_transform(
    [doc["title"] + " " + doc["text"] for doc in documents]
)

def retrieve_documents(question, top_k=3):
    query_vector = vectorizer.transform([question])
    scores = cosine_similarity(query_vector, document_matrix)[0]
    indices = scores.argsort()[::-1][:top_k]

    results = []
    for i in indices:
        if scores[i] > 0:
            item = documents[i].copy()
            item["relevance"] = float(scores[i])
            results.append(item)

    return results


for doc in retrieve_documents(
    "¿Qué diferencia hay entre un establecimiento y una empresa?"
):
    print(doc["title"], round(doc["relevance"], 3))
    print(doc["text"])
    print(doc["source"])
    print()

Qué es el DENUE 0.132
El Directorio Estadístico Nacional de Unidades Económicas, DENUE, ofrece información de establecimientos económicos en México. Incluye identificación, ubicación, actividad económica y estrato de personal ocupado. No debe interpretarse automáticamente como un registro de ingresos, ventas o utilidades.
https://www.inegi.org.mx/servicios/api_denue.html

Estratos de personal ocupado 0.091
La documentación de la API DENUE describe estratos de personal ocupado. Entre ellos están de 0 a 5, de 6 a 10, de 11 a 30, de 31 a 50, de 51 a 100, de 101 a 250 y de 251 personas en adelante. El estrato es una categoría de tamaño, no una medición de ventas ni de rentabilidad.
https://www.inegi.org.mx/servicios/api_denue.html

Búsqueda de establecimientos 0.069
El método BuscarEntidad permite buscar establecimientos por una condición y limitar la búsqueda a una entidad federativa. Los resultados pueden incluir nombre, actividad económica, ubicación y estrato de personal ocupado.
https

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Qwen está listo.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen está listo.


In [17]:
def format_evidence(question, api_evidence=None, top_k=3):
    retrieved = retrieve_documents(question, top_k=top_k)

    parts = []

    for doc in retrieved:
        parts.append(
            f"Documento: {doc['title']}\n"
            f"Contenido: {doc['text']}\n"
            f"Fuente: {doc['source']}"
        )

    if api_evidence is not None:
        parts.append(
            "Resultado de la consulta oficial DENUE:\n"
            + json.dumps(
                api_evidence,
                ensure_ascii=False,
                default=str
            )[:12000]
        )

    return "\n\n".join(parts), retrieved


def ask_qwen(question, api_evidence=None):
    evidence, retrieved = format_evidence(
        question,
        api_evidence=api_evidence
    )

    messages = [
        {
            "role": "system",
            "content": (
                "Eres un asistente público en español especializado "
                "en estadísticas económicas de México. Responde de forma "
                "clara y accesible. Usa los documentos y datos suministrados "
                "como evidencia. No inventes cifras, periodos, fuentes ni "
                "códigos geográficos. Si los datos no permiten responder, "
                "explícalo y pide la información faltante. Distingue "
                "establecimientos de empresas, trabajadores, ventas y "
                "utilidades. Para cualquier cifra, indica el periodo y "
                "alcance si están disponibles. Incluye los enlaces de "
                "las fuentes proporcionadas."
            )
        },
        {
            "role": "user",
            "content": (
                f"Pregunta: {question}\n\n"
                f"Evidencia disponible:\n{evidence}\n\n"
                "Redacta una respuesta basada en la evidencia."
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    sources = list(dict.fromkeys(
        doc["source"] for doc in retrieved
    ))

    return {
        "answer": answer,
        "sources": sources
    }


result = ask_qwen(
    "¿Qué información ofrece el DENUE sobre los negocios en México?"
)

print(result["answer"])
print("\nFuentes:")
for source in result["sources"]:
    print(source)

El Denúe, dirigido por la Secretaría de Economía del gobierno federal mexicano, ofrece una variedad de informaciones sobre los negocios en México. Aquí está una lista detallada de lo que puede encontrar:

1. **Identificación**: La información proporcionada por Denúe se centra en la identificación de los establecimientos económicos en México. Esto incluye detalles como la identificación del negocio, su ubicación geográfica y el tipo de actividad económica.

2. **Ubicación**: Denúe también proporciona información sobre la ubicación de estos establecimientos. Esto puede ser útil para entender dónde se encuentran los negocios y cómo se distribuyen entre diferentes áreas geográficas.

3. **Actividad Económica**: Denúe incluye información sobre la actividad económica de cada establecimiento. Esto puede ayudar a entender qué tipos de negocios son más comunes en diferentes regiones de México.

4. **Estándar de Personal Ocupado**: Denúe también proporciona información sobre el nivel de personal

In [15]:
STATE_CODES = {
    "aguascalientes": "01",
    "baja california": "02",
    "baja california sur": "03",
    "campeche": "04",
    "coahuila": "05",
    "colima": "06",
    "chiapas": "07",
    "chihuahua": "08",
    "ciudad de mexico": "09",
    "ciudad de méxico": "09",
    "durango": "10",
    "guanajuato": "11",
    "guerrero": "12",
    "hidalgo": "13",
    "jalisco": "14",
    "estado de mexico": "15",
    "estado de méxico": "15",
    "michoacan": "16",
    "michoacán": "16",
    "morelos": "17",
    "nayarit": "18",
    "nuevo leon": "19",
    "nuevo león": "19",
    "oaxaca": "20",
    "puebla": "21",
    "queretaro": "22",
    "querétaro": "22",
    "quintana roo": "23",
    "san luis potosi": "24",
    "san luis potosí": "24",
    "sinaloa": "25",
    "sonora": "26",
    "tabasco": "27",
    "tamaulipas": "28",
    "tlaxcala": "29",
    "veracruz": "30",
    "yucatan": "31",
    "yucatán": "31",
    "zacatecas": "32"
}

def get_live_count():
    data = denue_count(activity="0", geography="0", size="0")

    return {
        "tipo": "conteo_DENUE",
        "parametros": {
            "actividad_economica": "0",
            "area_geografica": "0",
            "estrato_personal": "0"
        },
        "datos": data,
        "fuente": "https://www.inegi.org.mx/servicios/api_denue.html",
        "fecha_consulta_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }


def get_state_search(term, state):
    normalized_state = state.strip().lower()

    if normalized_state not in STATE_CODES:
        return {
            "error": (
                "Estado no reconocido. Indica una entidad federativa "
                "de México."
            )
        }

    df = denue_search(
        term=term,
        state_code=STATE_CODES[normalized_state],
        start=1,
        end=10
    )

    # Return only fields useful for a public demonstration.
    useful_columns = [
        "Nombre",
        "Clase_actividad",
        "Estrato",
        "Ubicacion"
    ]

    available_columns = [
        col for col in useful_columns if col in df.columns
    ]

    return {
        "tipo": "busqueda_DENUE",
        "termino": term,
        "estado": state,
        "registros_de_muestra": df[available_columns].to_dict(
            orient="records"
        ),
        "nota": (
            "Muestra de registros devueltos por la búsqueda; "
            "no representa necesariamente el total de establecimientos."
        ),
        "fuente": "https://www.inegi.org.mx/servicios/api_denue.html",
        "fecha_consulta_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

In [16]:
import re
import unicodedata

def normalize_text(text):
    text = unicodedata.normalize("NFD", text.lower())
    return "".join(
        ch for ch in text
        if unicodedata.category(ch) != "Mn"
    )


def chatbot(question):
    q = normalize_text(question)

    # Route explicit questions about the national establishment count.
    if (
        "cuantos establecimientos hay en mexico" in q
        or "cuantos negocios hay en mexico" in q
        or "total de establecimientos en mexico" in q
    ):
        evidence = get_live_count()
        return ask_qwen(
            question,
            api_evidence=evidence
        )

    # Route a simple business search by term and state.
    match = re.search(
        r"(?:busca|buscar|encuentra|muestra)\s+"
        r"(.+?)\s+(?:en|del estado de)\s+"
        r"([a-zA-ZáéíóúÁÉÍÓÚñÑ ]+?)\s*[?.!]*$",
        question.strip(),
        flags=re.IGNORECASE
    )

    if match:
        term = match.group(1).strip(" ?.!¿")
        state = match.group(2).strip(" ?.!¿")

        evidence = get_state_search(term, state)

        if "error" in evidence:
            return {
                "answer": evidence["error"],
                "sources": []
            }

        return ask_qwen(
            question,
            api_evidence=evidence
        )

    # For questions about definitions and methodology, use RAG.
    return ask_qwen(question)


response = chatbot(
    "¿Qué información ofrece el DENUE sobre los negocios en México?"
)

print(response["answer"])
print("\nFuentes:", response["sources"])

El Denúe, dirigido por la Secretaría de Economía del gobierno federal mexicano, ofrece una variedad de informaciones sobre los negocios en México. Aquí está una lista detallada de lo que puede encontrar:

1. **Identificación**: La información proporcionada por Denúe se centra en la identificación de los establecimientos económicos en México. Esto incluye detalles como la identificación del negocio, su ubicación geográfica y el tipo de actividad económica.

2. **Ubicación**: Denúe también proporciona información sobre la ubicación de estos establecimientos. Esto puede ser útil para entender dónde se encuentran los negocios y cómo se distribuyen entre diferentes áreas geográficas.

3. **Actividad Económica**: Denúe incluye información sobre la actividad económica de cada establecimiento. Esto puede ayudar a entender qué tipos de negocios son más comunes en diferentes regiones de México.

4. **Estándar de Personal Ocupado**: Denúe también proporciona información sobre el nivel de personal

In [18]:
test_questions = [
    "¿Qué es el DENUE?",
    "¿Qué diferencia hay entre una empresa y un establecimiento?",
    "¿Qué significa el estrato de personal ocupado?",
    "¿Cuántos establecimientos hay en México?",
    "¿Qué información debo revisar antes de comparar negocios entre estados?"
]

for question in test_questions:
    print("\n" + "=" * 70)
    print("PREGUNTA:", question)

    result = chatbot(question)
    print("\nRESPUESTA:", result["answer"])
    print("\nFUENTES:", result["sources"])


PREGUNTA: ¿Qué es el DENUE?

RESPUESTA: El Denúe es un sistema de estadística nacional que proporciona información sobre las unidades económicas en México. Este sistema incluye información sobre la identidad, ubicación, actividad económica y estrato de personal ocupado. Sin embargo, este sistema no se puede interpretar como un registro de ingresos, ventas o utilidades.

Para obtener información más precisa sobre el Denúe, te recomiendo consultar la documentación oficial del INEGI (Instituto Nacional de Estadística y Geografía) o buscar en línea la documentación específica de la API denue.

FUENTES: ['https://www.inegi.org.mx/servicios/api_denue.html']

PREGUNTA: ¿Qué diferencia hay entre una empresa y un establecimiento?

RESPUESTA: Una empresa es un ente jurídico que se dedica al comercio, servicios o producción de bienes. Es una organización con un propósito específico para generar ingresos. Por otro lado, un establecimiento económico es un lugar donde se realiza la actividad económ

In [19]:
!pip -q install gradio

In [20]:
import gradio as gr

def respond(message, history):
    try:
        result = chatbot(message)
        answer = result["answer"]

        if result.get("sources"):
            answer += "\n\nFuentes:\n" + "\n".join(
                result["sources"]
            )

        return answer

    except Exception:
        return (
            "No pude completar la consulta en este momento. "
            "Intenta nuevamente más tarde."
        )


demo = gr.ChatInterface(
    fn=respond,
    title="INEGI Qwen — Estadísticas de negocios",
    description=(
        "Consulta información sobre establecimientos económicos "
        "en México. Las cifras dependen de los datos disponibles "
        "en las fuentes oficiales."
    )
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f4b2bb66bb56bccfd9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
